In [ ]:
# LLaVA-NeXT 데이터셋 실습 코드: 이미지와 대화로 AI 분석가 되기! 🕵️‍♀️
# ---------------------------------------------------------------------------
# ✨ 이 데이터셋은 LLaVA-NeXT라는 최첨단 모델의 학습에 사용된 '멀티모달' 데이터입니다.
# ✨ 즉, '이미지'와 '텍스트 대화'가 결합된, AI가 세상을 배우는 방식을 담고 있어요!
# ✨ 목표: 이 복잡한 데이터를 분석하여, AI가 어떤 정보를 추출하는지 탐색해 볼 거예요.
# ---------------------------------------------------------------------------

import numpy as np
from datasets import load_dataset
from PIL import Image
import random
import io # 이미지를 메모리에서 처리하기 위함

# --- 환경 설정 ---
DATASET_NAME = "lmms-lab/LLaVA-NeXT-Data"
SAMPLE_COUNT = 5 # 실습을 위해 상위 5개 샘플만 사용합니다!

print("==============================================================")
print("🧠 AI 코딩 튜터와 함께하는 멀티모달 데이터 탐색 시간! 💡")
print("==============================================================")


# 1. 데이터셋 로드 전략 (스트리밍 먼저 시도!)
print(f"🔍 1. 데이터셋 '{DATASET_NAME}'을 로드합니다. (최대 {SAMPLE_COUNT}개만 분석)")

try:
    # 스트리밍 모드로 로드 시도 (대용량 데이터셋에 최적)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 스트리밍 모드 로드 성공! 메모리 효율적입니다. (가장 좋아요!)")
except Exception:
    # 스트리밍이 어려운 경우, 적은 수의 데이터만 로드하여 진행
    print("⚠️ 경고: 스트리밍 모드 로드에 실패했습니다. 작은 배치를 로드하여 진행합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train')
    except Exception as e:
        print(f"❌ 데이터셋 로드에 심각한 오류가 발생했습니다: {e}")
        dataset = None

if dataset is None:
    print("😭 코드를 실행할 수 없습니다. 데이터셋 로드를 다시 확인해 주세요.")
    exit()


# 2. 데이터셋 반복자(Iterator) 설정 (Streaming/Non-streaming 호환성 확보)
# 'len()'을 사용하지 않고, take()를 사용하여 안전하게 샘플을 추출합니다.

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    sample_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)
    # 이 경우에는 list()로 변환 후 처음 N개를 사용합니다.
    sampled_dataset = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))
    sample_iterator = sampled_dataset


# 3. 데이터 분석 및 추론 시뮬레이션 (실습 메인 구간)
print("\n--------------------------------------------------------------")
print(f"🔎 2. 상위 {min(SAMPLE_COUNT, 10)}개 샘플을 분석하며 AI의 사고 과정을 따라가 봅시다!")
print("--------------------------------------------------------------")

# 샘플 데이터 목록을 생성합니다. (최대 10개까지 처리)
sample_data_list = []
for i, sample in enumerate(sample_iterator):
    if i >= 10: # 최대 10개 샘플만 분석합니다.
        break
    sample_data_list.append(sample)

# 핵심 분석 루프 시작
for i, sample in enumerate(sample_data_list):
    print(f"\n=== [Sample #{i+1} / {len(sample_data_list)}] ✨ 분석 시작 ===")

    # 3-1. 이미지 데이터 분석 (Vision Part)
    image = sample.get('image')
    if image is not None:
        print(f"🖼️ [IMAGE ANALYSIS]: 이미지를 감지했습니다! (type: {type(image).__name__})")
        
        # 이미지 처리를 위해 PIL.Image.Image 객체로 간주하고 크기 확인
        try:
            # Image object는 .size를 사용합니다.
            image_width, image_height = image.size 
            print(f"   -> 📐 크기 확인: {image_width}px x {image_height}px")
        except Exception as e:
            print(f"   -> ⚠️ 이미지 크기 확인 실패 (오류: {e}). 데이터 구조를 확인해 보세요.")

        # 실제 이미지 데이터가 넘파이 배열 형태라면 (추가 분석 시)
        # image_np = np.array(image)
        # print(f"   -> 🔢 NumPy 배열 변환 가능 여부 확인 (Shape): {image_np.shape}")


    # 3-2. 대화/지시사항 분석 (NLP/Instruction Part)
    conversations = sample.get('conversations')
    if conversations and isinstance(conversations, list):
        
        # 대화 기록에서 최종 사용자 질문을 추출하는 함수 (가장 중요한 정보!)
        user_query = "질문 없음"
        for turn in conversations:
            if turn.get('from', '').lower() == 'user':
                user_query = turn.get('value', '')
        
        print(f"💬 [CONVERSATION ANALYSIS]: 대화 흐름을 분석했습니다.")
        print(f"   -> 🗣️ 추출된 최종 사용자 질문 (Query): '{user_query[:50]}...'")
    else:
        print("💬 [CONVERSATION ANALYSIS]: 대화 기록을 찾을 수 없습니다.")

    
    # 3-3. 종합 분석 및 가상 추론 (The Fun Part!)
    print("\n✨ [🧠 튜터의 한 줄 코딩 AI 추론]:")
    
    if "무엇" in user_query and "어" in user_query:
        print("   💡 역할: 이 샘플은 '무엇'에 대한 질문을 하는 VQA (Visual Question Answering) 유형입니다.")
    elif "무엇을" in user_query or "설명" in user_query:
        print("   💡 역할: 이 샘플은 '설명' 또는 '정보 요청' 유형입니다. (Captioning/Description)")
    elif user_query:
        print("   💡 역할: 일반적인 지시 따르기 (Instruction Following) 샘플로 보입니다.")
    else:
        print("   💡 역할: 대화 내용이 불분명하여, 추가적인 전처리(Pre-processing)가 필요할 수 있습니다.")
        
    print("--------------------------------------------------------------")


# 4. 마무리 코딩 스니펫
print("\n==============================================================")
print("🎉 실습 완료! 코딩 실력을 점검할 시간이에요.")
print("==============================================================")
print("# ✅ 기억하세요! LLaVA-NeXT는 'Image + [Question]' 구조로 학습됩니다.")
print("만약 이 코드를 실제 LLM 파이프라인에 적용한다면, 마지막 분석 과정(3-3)을")
print("실제 LLM의 프롬프트 엔지니어링에 활용할 거예요!")
print("# 예시: prompt = f\"[IMAGE] {image} \n\n[USER QUESTION] {user_query}\"")